# Register list functions

Copyright 2024 Google LLC. Adapted under Apache-2.0 from locked Qualtran v0.7.0, commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable
from typing import cast

from IPython import display as notebook_display
from sympy import Symbol

from examples.algorithms.field_addition_catalog import field_signature_svg
from examples.algorithms.register_sorted_duplicates import main as execute_example
from flagquantum.compiler._component_trace.call_graph import (
    modeled_call_graph,
    symbolic_call_graph,
)
from flagquantum.compiler._component_trace.resource_model import (
    estimate_component_resources,
)
from flagquantum.compiler.registers.sorting.in_place_sort_declaration import (
    in_place_sort_signature,
    source_in_place_sort_cost,
)
from flagquantum.compiler.registers.sorting.sorted_duplicates import (
    sorted_duplicates_signature,
)
from flagquantum.compiler.registers.sorting.symmetric_difference import (
    symmetric_difference_ir,
    symmetric_difference_requirements,
)
from flagquantum.compiler.registers.sorting.symmetric_difference_declaration import (
    source_symmetric_difference_cost,
    symmetric_difference_signature,
)
from flagquantum.core.ir.program.register_types import unsigned_register_type
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


## In-place sorting
Sort a list of $\ell$ numbers in place using $\ell \log \ell$ ancilla bits.

Applies the map:
$$
    |x_1, x_2, \ldots, x_l\rangle
    |0^{\ell \log \ell}\rangle
    \mapsto
    |x_{\pi_1}, x_{\pi_2}, \ldots, x_{\pi_\ell})\rangle
    |\pi_1, \pi_2, \ldots, \pi_\ell\rangle
$$
where $x_{\pi_1} \le x_{\pi_2} \ldots \le x_{\pi_\ell}$ is the sorted list,
and the ancilla are entangled.

To apply this, we first use any sorting algorithm to output the sorted list
in a clean register. And then use the following algorithm from Lemma 4.12 of Ref [1]
that applies the map:

$$
    |x_1, ..., x_l\rangle|x_{\pi(1)}, ..., x_{\pi(l)})\rangle
    \mapsto
    |x_l, ..., x_l\rangle|\pi(1), ..., \pi(l))\rangle
$$

where $x_i \in [n]$ and $\pi(i) \in [l]$.
This second algorithm (Lemma 4.12) has two steps, each with $l^2$ comparisons:
1. compute `pi(1) ... pi(l)` given `x_1 ... x_l` and `x_{pi(1)} ... x{pi(l)}`.
1. (un)compute `x_{pi(1)} ... x{pi(l)}` using `pi(1) ... pi(l)` given `x_1 ... x_l`.

#### Parameters
 - `l`: number of elements in the list
 - `dtype`: type of each element to store `[n]`. 

#### Registers
 - `input`: the entire input as a single register
 - `ancilla`: the generated (entangled) register storing `pi`. 

#### References
 - [Quartic quantum speedups for planted inference](https://arxiv.org/abs/2406.19378v1). Lemma 4.12. Eq. 122.


In [ ]:
sort_signature = in_place_sort_signature(4, unsigned_register_type(3))
sort_model = source_in_place_sort_cost(4, 3)
# The original source section only imports this declaration.


## Symmetric difference
Given two sorted sets $S, T$ of unique elements, compute their symmetric difference.

This accepts an integer `n_diff`, and marks a flag qubit if the symmetric difference
set is of size exactly `n_diff`. If the flag is marked (1), then the output of `n_diff`
numbers is the symmetric difference, otherwise it may be arbitrary.

#### Parameters
 - `n_lhs`: number of elements in $S$
 - `n_rhs`: number of elements in $T$
 - `n_diff`: expected number of elements in the difference $S \Delta T$.
 - `dtype`: type of each element. 

#### Registers
 - `S`: list of `n_lhs` numbers.
 - `T`: list of `n_rhs` numbers.
 - `diff`: output register of `n_diff` numbers.
 - `flag`: XORed when the symmetric-difference cardinality matches the requested output size. 

#### References
 - [Quartic quantum speedups for planted inference](https://arxiv.org/abs/2406.19378v1). Theorem 4.17, proof para 3, page 38.


In [ ]:
# Metadata and model come from the native sorting declaration owner.


### Example Instances

In [ ]:
symm_diff = source_symmetric_difference_cost(4, 2, 4, 4)
symmetric_signature = symmetric_difference_signature(4, 2, 4, unsigned_register_type(4))


#### Graphical Signature

In [ ]:
display(
    SVG(
        field_signature_svg(
            (
                (
                    "symmetric difference (4+2 words, uint[4], 4 outputs)",
                    symmetric_signature,
                    {},
                ),
            )
        )
    )
)


### Call Graph

In [ ]:
symmetric_graph = modeled_call_graph(symm_diff, max_depth=1)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                symmetric_graph.root,
                symmetric_graph.direct_calls,
                title=lambda component: component.name,
            )
        )
    )
)
display({child.name: count for child, count in symmetric_graph.leaf_counts.items()})


## Sorted duplicate detection
Given a sorted list of `l` numbers, check if it contains any duplicates.

Produces a single qubit which is `1` if there are duplicates, and `0` if all are disjoint.
It compares every adjacent pair, and therefore uses `l - 1` comparisons.
It then uses a single MCX on `l - 1` bits gate to compute the flag.

#### Parameters
 - `l`: number of elements in the list
 - `dtype`: type of each element to store `[n]`. 

#### Registers
 - `xs`: a list of `l` registers of `dtype`.
 - `flag`: single qubit. Value is flipped if the input list has duplicates, otherwise stays same. 

#### References
 - [Quartic quantum speedups for planted inference](https://arxiv.org/abs/2406.19378v1). Lemma 4.12. Eq. 122.


In [ ]:
# The duplicate detector keeps its own signature and physical implementation.


### Example Instances

In [ ]:
n = Symbol("n")
has_duplicates_symb = sorted_duplicates_signature(4, unsigned_register_type(n))


In [ ]:
has_duplicates = sorted_duplicates_signature(4, unsigned_register_type(3))


#### Graphical Signature

In [ ]:
display(
    SVG(
        field_signature_svg(
            (
                ("sorted duplicates (4 words, uint[n])", has_duplicates_symb, {}),
                ("sorted duplicates (4 words, uint[3])", has_duplicates, {}),
            )
        )
    )
)


### Call Graph

In [ ]:
# Direct source declarations, shown before physical decomposition.
duplicate_calls = {
    "Half less than": 3,
    "Half less than adjoint": 3,
    "Three-control X": 1,
    "X": 1,
}
duplicate_graph = symbolic_call_graph(
    "Sorted duplicates (4 words)",
    {"Sorted duplicates (4 words)": duplicate_calls},
    max_depth=1,
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(duplicate_graph.root, duplicate_graph.direct_calls)
        )
    )
)
display(duplicate_graph.leaf_counts)


## Execution and separate resource accounts

The existing duplicate example executes `circuit.append(op)`, verifies every complex amplitude and restores clean work. The source alias/erasure and ignored-control callback defects remain in the file audit.

The symmetric-difference source root has no decomposition. Its concrete graph above omits compaction and impossible-parity rejection. Native executable compaction and measurement-assisted cleanup have their own primitive inventory below. Its coherent spectator and all small-history evidence are retained in the existing module tests. Protected execution admission remains open.


In [ ]:
execute_example()
work_width, classical_width = symmetric_difference_requirements(4, 2, 4, 4)
words = tuple(tuple(range(4 * i, 4 * (i + 1))) for i in range(10))
flag = 40
work = tuple(range(41, 41 + work_width))
gates = symmetric_difference_ir(
    words[:4], words[4:6], words[6:], flag, work, tuple(range(classical_width))
)
measured_primitives = dict(Counter(g.name for g in gates))
display(measured_primitives)
display(estimate_component_resources(symm_diff))
